# Human Activity Recognition — Demo Notebook
> Multi-Modal Deep Learning · Explainability · Robustness Analysis  
> M.Eng. Capstone · University of Connecticut · May 2026 · Shubham Maheshwari

This notebook walks the full HAR pipeline using the modular `src/har` package.

| # | Section |
|---|---|
| 1 | Setup & GPU check |
| 2 | Load UCI HAR data |
| 3 | Define all 7 models |
| 4 | Train |
| 5 | Evaluate (accuracy, F1, confusion matrices, ROC) |
| 6 | Robustness under sensor degradation |
| 7 | SHAP explainability |
| 8 | Self-supervised pre-training |
| 9 | Knowledge distillation — TinyHAR |
| 10 | Fall detection case study |
| 11 | WISDM cross-dataset generalisation |
| 12 | Final summary |


## 1 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
sys.path.insert(0, os.path.abspath("../"))

from configs.config import DEVICE, CLASS_NAMES
import torch

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 2 · Load Data
Subject-based 70/30 split — 21 train subjects, 9 test subjects, no overlap.

In [ ]:
from har.data import load_uci_har, make_dataloaders

data = load_uci_har()
Xf_tr, Xf_te = data["Xf_tr"], data["Xf_te"]
Xt_tr, Xt_te = data["Xt_tr"], data["Xt_te"]
y_tr,  y_te  = data["y_tr"],  data["y_te"]
scaler        = data["scaler"]

dl_feat_tr, dl_feat_te, dl_mm_tr, dl_mm_te = make_dataloaders(data)
print("DataLoaders ready.")


## 3 · Define Models

In [ ]:
from har.models import (
    SimpleMLP, RegMLP, DeepMLP,
    FusionNet, GatedFusionNet,
    BiLSTM, CNNLSTMHybrid,
    TinyHAR,
    SSLEncoder, SSLPretrainModel, SSLClassifier,
)

catalog = [
    ("SimpleMLP",      SimpleMLP(),      "Feature-only"),
    ("RegMLP",         RegMLP(),         "Feature-only"),
    ("DeepMLP",        DeepMLP(),        "Feature-only"),
    ("FusionNet",      FusionNet(),      "Multi-modal"),
    ("GatedFusionNet", GatedFusionNet(), "Multi-modal"),
    ("BiLSTM",         BiLSTM(),         "Advanced"),
    ("CNN-LSTM",       CNNLSTMHybrid(),  "Advanced"),
]
print(f"  {'Model':<18} {'Params':>12}  Category")
print("  " + "-" * 46)
for name, m, cat in catalog:
    p = sum(x.numel() for x in m.parameters())
    print(f"  {name:<18} {p:>12,}  {cat}")


## 4 · Train

In [ ]:
from har.training import train_model
from har.evaluation import plot_history
from configs.config import (
    EPOCHS_MLP, EPOCHS_FUSION, EPOCHS_SEQ,
    LR_MLP, LR_FUSION, LR_SEQ,
)

m_simple  = SimpleMLP()
m_reg     = RegMLP()
m_deep    = DeepMLP()
m_fusion  = FusionNet()
m_gated   = GatedFusionNet()
m_bilstm  = BiLSTM()
m_cnnlstm = CNNLSTMHybrid()

h_simple  = train_model(m_simple,  dl_feat_tr, epochs=EPOCHS_MLP,    lr=LR_MLP,    name="SimpleMLP")
h_reg     = train_model(m_reg,     dl_feat_tr, epochs=EPOCHS_MLP,    lr=LR_MLP,    name="RegMLP")
h_deep    = train_model(m_deep,    dl_feat_tr, epochs=EPOCHS_MLP,    lr=LR_MLP,    name="DeepMLP")
h_fusion  = train_model(m_fusion,  dl_mm_tr,   epochs=EPOCHS_FUSION, lr=LR_FUSION, name="FusionNet",   multimodal=True)
h_gated   = train_model(m_gated,   dl_mm_tr,   epochs=EPOCHS_FUSION, lr=LR_FUSION, name="GatedFusion", multimodal=True)
h_bilstm  = train_model(m_bilstm,  dl_mm_tr,   epochs=EPOCHS_SEQ,    lr=LR_SEQ,    name="BiLSTM",      multimodal=True)
h_cnnlstm = train_model(m_cnnlstm, dl_mm_tr,   epochs=EPOCHS_SEQ,    lr=LR_SEQ,    name="CNN-LSTM",    multimodal=True)

for name, h in [("SimpleMLP", h_simple), ("FusionNet", h_fusion), ("CNN-LSTM", h_cnnlstm)]:
    plot_history(h, name)


## 5 · Evaluate

In [ ]:
from har.evaluation import report, plot_cm, plot_accuracy_bar

acc_simple,  yt_s,  yp_s  = report(m_simple,  dl_feat_te, "SimpleMLP")
acc_reg,     yt_r,  yp_r  = report(m_reg,     dl_feat_te, "RegMLP")
acc_deep,    yt_d,  yp_d  = report(m_deep,    dl_feat_te, "DeepMLP")
acc_fusion,  yt_f,  yp_f  = report(m_fusion,  dl_mm_te,   "FusionNet",      multimodal=True)
acc_gated,   yt_g,  yp_g  = report(m_gated,   dl_mm_te,   "GatedFusionNet", multimodal=True)
acc_bilstm,  yt_l,  yp_l  = report(m_bilstm,  dl_mm_te,   "BiLSTM",         multimodal=True)
acc_cnnlstm, yt_cl, yp_cl = report(m_cnnlstm, dl_mm_te,   "CNN-LSTM",       multimodal=True)

names_all = ["SimpleMLP","RegMLP","DeepMLP","FusionNet","GatedFusion","BiLSTM","CNN-LSTM"]
accs_all  = [acc_simple, acc_reg, acc_deep, acc_fusion, acc_gated, acc_bilstm, acc_cnnlstm]
plot_accuracy_bar(names_all, accs_all)


In [ ]:
for yt, yp, name in [(yt_s, yp_s, "SimpleMLP"),
                     (yt_f, yp_f, "FusionNet"),
                     (yt_cl, yp_cl, "CNN-LSTM")]:
    plot_cm(yt, yp, f"{name} — Confusion Matrix (9 unseen subjects)")


## 6 · Robustness Analysis

In [ ]:
from har.evaluation import robustness_eval
from har.evaluation.plots import plot_robustness

y_te_np = y_te.numpy()
rob = {
    "SimpleMLP":   robustness_eval(m_simple,  Xf_te, y_te_np),
    "RegMLP":      robustness_eval(m_reg,     Xf_te, y_te_np),
    "DeepMLP":     robustness_eval(m_deep,    Xf_te, y_te_np),
    "FusionNet":   robustness_eval(m_fusion,  Xf_te, y_te_np, multimodal=True, Xt=Xt_te),
    "GatedFusion": robustness_eval(m_gated,   Xf_te, y_te_np, multimodal=True, Xt=Xt_te),
    "BiLSTM":      robustness_eval(m_bilstm,  Xf_te, y_te_np, multimodal=True, Xt=Xt_te),
    "CNN-LSTM":    robustness_eval(m_cnnlstm, Xf_te, y_te_np, multimodal=True, Xt=Xt_te),
}
plot_robustness(rob)


## 7 · SHAP Explainability

In [ ]:
from har.explainability import run_shap_analysis
shap_vals = run_shap_analysis(m_deep, Xf_tr, Xf_te)


## 8 · Self-Supervised Pre-Training

In [ ]:
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from configs.config import EPOCHS_SSL, EPOCHS_FT, LR_SSL, LR_FT

ssl_model  = SSLPretrainModel().to(DEVICE)
ssl_optim  = optim.AdamW(ssl_model.parameters(), lr=LR_SSL, weight_decay=1e-4)
ssl_loader = DataLoader(TensorDataset(Xt_tr), batch_size=256, shuffle=True, drop_last=True)

for ep in range(EPOCHS_SSL):
    ssl_model.train(); ep_loss = 0.0
    for (ts_b,) in ssl_loader:
        ts_b = ts_b.to(DEVICE)
        recon, orig = ssl_model(ts_b)
        loss = F.mse_loss(recon, orig)
        ssl_optim.zero_grad(); loss.backward(); ssl_optim.step()
        ep_loss += loss.item()
    if (ep + 1) % 5 == 0:
        print(f"  ep{ep+1}/{EPOCHS_SSL}  recon_loss={ep_loss/len(ssl_loader):.4f}")

ssl_clf      = SSLClassifier(ssl_model.encoder).to(DEVICE)
h_ssl        = train_model(ssl_clf, dl_mm_tr, multimodal=True, name="SSL-Finetuned", epochs=EPOCHS_FT, lr=LR_FT)
acc_ssl, _,_ = report(ssl_clf, dl_mm_te, "SSL Pre-trained", multimodal=True)

scratch_clf      = SSLClassifier(SSLEncoder())
h_scratch        = train_model(scratch_clf, dl_mm_tr, multimodal=True, name="No-SSL", epochs=EPOCHS_FT, lr=1e-3)
acc_scratch, _,_ = report(scratch_clf, dl_mm_te, "No-SSL Scratch", multimodal=True)

ssl_gain = acc_ssl - acc_scratch
print(f"SSL benefit: +{ssl_gain:.2f}pp")


## 9 · Knowledge Distillation — TinyHAR

In [ ]:
from har.training import train_kd
from configs.config import EPOCHS_KD, LR_KD

t_params = sum(p.numel() for p in m_fusion.parameters())
s_params = sum(p.numel() for p in TinyHAR().parameters())
print(f"Teacher: {t_params:,} params  |  Student: {s_params:,} params  ({t_params//s_params}x smaller)")

tiny_kd      = TinyHAR()
h_kd         = train_kd(m_fusion, tiny_kd, dl_feat_tr, epochs=EPOCHS_KD, lr=LR_KD)
acc_kd, _,_  = report(tiny_kd, dl_feat_te, "TinyHAR (KD)")

tiny_sc      = TinyHAR()
h_sc         = train_model(tiny_sc, dl_feat_tr, name="TinyHAR-Scratch", epochs=EPOCHS_KD)
acc_sc, _,_  = report(tiny_sc, dl_feat_te, "TinyHAR (Scratch)")

print(f"Teacher {acc_fusion:.2f}%  |  KD {acc_kd:.2f}%  |  Scratch {acc_sc:.2f}%  |  KD gain +{acc_kd-acc_sc:.2f}pp")


## 10 · Fall Detection Case Study

In [ ]:
from har.experiments.fall_detection import run_fall_detection_case_study

fall_res = run_fall_detection_case_study([
    ("SimpleMLP",  m_simple,  dl_feat_te, False),
    ("DeepMLP",    m_deep,    dl_feat_te, False),
    ("FusionNet",  m_fusion,  dl_mm_te,   True),
    ("BiLSTM",     m_bilstm,  dl_mm_te,   True),
    ("CNN-LSTM",   m_cnnlstm, dl_mm_te,   True),
    ("TinyHAR-KD", tiny_kd,   dl_feat_te, False),
], y_te.numpy())


## 11 · WISDM Cross-Dataset Generalisation

In [ ]:
from har.experiments.wisdm import run_wisdm_generalisation

wisdm_res = run_wisdm_generalisation(
    feature_models=[("SimpleMLP", m_simple), ("DeepMLP", m_deep)],
    scaler=scaler,
    Xf_te=Xf_te,
    best_acc_uci=max(accs_all),
)


## 12 · Final Summary

In [ ]:
import torch, os

print("=" * 65)
print("  CAPSTONE PROJECT — FINAL RESULTS SUMMARY")
print("=" * 65)

rows = [
    ("SimpleMLP",   acc_simple,  "Feature-only",  "Baseline"),
    ("RegMLP",      acc_reg,     "Feature-only",  "+ Dropout"),
    ("DeepMLP",     acc_deep,    "Feature-only",  "+ BatchNorm, depth"),
    ("FusionNet",   acc_fusion,  "Multi-modal",   "CNN + feature concat"),
    ("GatedFusion", acc_gated,   "Multi-modal",   "Learnable gate alpha"),
    ("BiLSTM",      acc_bilstm,  "Advanced",      "Bidirectional LSTM"),
    ("CNN-LSTM",    acc_cnnlstm, "Advanced",      "CNN -> LSTM"),
]

print(f"\n  {'Model':<16} {'Accuracy':>9}  {'Category':<14}  Notes")
print("  " + "─" * 60)
best_acc = max(a for _, a, _, _ in rows)
for name, acc, cat, note in sorted(rows, key=lambda x: -x[1]):
    tag = "  BEST" if acc == best_acc else ""
    print(f"  {name:<16} {acc:>8.2f}%  {cat:<14}  {note}{tag}")

ssl_gain = acc_ssl - acc_scratch
print(f"\nSSL Pre-training       : {acc_ssl:.2f}% vs {acc_scratch:.2f}% scratch (+{ssl_gain:.2f}pp)")
print(f"Knowledge Distillation : TinyHAR {acc_kd:.2f}% (37x smaller than FusionNet)")

best_entry = max(
    [("SimpleMLP",m_simple,acc_simple),("RegMLP",m_reg,acc_reg),
     ("DeepMLP",m_deep,acc_deep),("FusionNet",m_fusion,acc_fusion),
     ("GatedFusion",m_gated,acc_gated),("BiLSTM",m_bilstm,acc_bilstm),
     ("CNN-LSTM",m_cnnlstm,acc_cnnlstm)],
    key=lambda x: x[2]
)
best_name, best_model, _ = best_entry
ckpt = os.path.join(os.path.abspath("../checkpoints"), f"best_HAR_{best_name}.pth")
torch.save(best_model.state_dict(), ckpt)
print(f"\nBest model saved -> {ckpt}")
print("=" * 65)
